# Hull Tactical Market Prediction - Working Notebook

Portable version of the DDA3020 course submission. The unmodified notebook is in `notebooks/original/`. See `docs/ORGANIZATION.md` for runtime changes and `docs/KNOWN_LIMITATIONS.md` for leakage and evaluation caveats. Historical scores have not been reproduced.


## 1. Configure Paths and Environment

Install `requirements.txt` before executing. No automatic package installation or pickle discovery is performed.


In [ ]:
import os
from pathlib import Path

# Find this repository whether Jupyter starts at its root or notebooks/.
_here = Path.cwd().resolve()
REPO_ROOT = next((p for p in [_here, *_here.parents]
                  if (p / "requirements.in").is_file() and (p / "notebooks").is_dir()), _here)
KAGGLE_DATA = Path("/kaggle/input/hull-tactical-market-prediction")
DEFAULT_DATA = KAGGLE_DATA if KAGGLE_DATA.is_dir() else REPO_ROOT / "data"
TRAIN_PATH = Path(os.getenv("HULL_TRAIN_PATH", str(DEFAULT_DATA / "train.csv"))).expanduser().resolve()
TEST_PATH = Path(os.getenv("HULL_TEST_PATH", str(DEFAULT_DATA / "test.csv"))).expanduser().resolve()
OUTPUT_DIR = Path(os.getenv("HULL_OUTPUT_DIR", "/kaggle/working" if KAGGLE_DATA.is_dir()
                            else str(REPO_ROOT / "artifacts"))).expanduser().resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
N_JOBS = int(os.getenv("HULL_N_JOBS", "1"))
for label, path in [("training", TRAIN_PATH), ("test", TEST_PATH)]:
    if not path.is_file():
        raise FileNotFoundError(f"Missing {label} CSV: {path}. See README.md for data setup.")
print(f"Training: {TRAIN_PATH}\nTest: {TEST_PATH}\nOutputs: {OUTPUT_DIR}")


In [ ]:
# Dependencies are installed once in the environment, not during a run.
import sys
print(f"Python: {sys.version.split()[0]}")


## 2. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import pickle
import warnings
# Keep warnings visible so incompatible schemas or numerical issues are observable.

from sklearn.preprocessing import StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import Ridge
import lightgbm as lgb
import xgboost as xgb


# Random Seed
print("✅ Libraries imported successfully!")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [ ]:
# === Train Optimized Model
print("="*80)
print("⚙️ Train Optimized Model (Enhanced Version)")
print("="*80)

# TRAIN_PATH comes from the configuration cell.

print(f"Using training data path: {TRAIN_PATH}")

# Read training data
try:
    df_train = pd.read_csv(TRAIN_PATH)
    print(f"Training set shape: {df_train.shape}")
    print(f"Training set columns: {df_train.columns.tolist()[:20]}")
except Exception as e:
    print(f"❌ Unable to read training data: {e}")
    raise

# Automatically detect target column
possible_targets = ['forward_returns', 'market_forward_excess_returns', 'target', 'lagged_forward_returns']
target_col = os.getenv('HULL_TARGET_COLUMN') or next((c for c in possible_targets if c in df_train.columns), None)
if target_col is None or target_col not in df_train.columns:
    raise ValueError(f"Cannot find target column in training set, candidates: {possible_targets}")

print(f"Using target column: {target_col}")

# Clean and prepare features
drop_cols = [c for c in ['row_id', 'batch_id', 'date_id', 'is_scored'] if c in df_train.columns]
X_all = df_train.drop(columns=drop_cols + [target_col])
y_all = df_train[target_col]
if len(df_train) < 10 or not np.isfinite(y_all.to_numpy(dtype=float)).all():
    raise ValueError('Training requires at least 10 rows and a finite numeric target.')
raw_feature_names = X_all.columns.tolist()
raw_feature_medians = X_all.median(numeric_only=True).reindex(raw_feature_names).fillna(0.0)
imputer = None

print(f"\nTarget Variable Statistics:")
print(f"  Mean: {y_all.mean():.6f}")
print(f"  Std Dev: {y_all.std():.6f}")
print(f"  Range: [{y_all.min():.6f}, {y_all.max():.6f}]")

# === Improvement 1: Use IterativeImputer for Advanced Missing Value Filling ===
print("\n🔧 Handling missing values...")
missing_count = X_all.isnull().sum().sum()
if missing_count > 0:
    print(f"Detected {missing_count} missing values, using IterativeImputer for filling")
    imputer = IterativeImputer(
        estimator=Ridge(alpha=1.0),
        max_iter=10,
        random_state=RANDOM_SEED,
        verbose=0,
        keep_empty_features=True
    )
    X_all = pd.DataFrame(
        imputer.fit_transform(X_all),
        columns=X_all.columns,
        index=X_all.index
    )
    print(f"✅ Missing value filling completed")
else:
    print("No missing values")

# === Improvement 2: Smarter Feature Selection ===
print("\n🔧 Smart Feature Selection...")

# 1. Remove low variance features
from sklearn.feature_selection import VarianceThreshold
var_threshold = VarianceThreshold(threshold=0.001)  # Lower threshold to keep more features
X_all = pd.DataFrame(var_threshold.fit_transform(X_all), 
                     columns=X_all.columns[var_threshold.get_support()],
                     index=X_all.index)

# 2. Correlation-based feature selection (with target variable)
from sklearn.feature_selection import SelectKBest, f_regression
correlations = X_all.corrwith(y_all).abs().sort_values(ascending=False)
print(f"Top 10 Correlated Features: {correlations.head(10).to_dict()}")

# Keep more features (at least 60)
k_features = min(max(60, len(X_all.columns) * 2 // 3), len(X_all.columns))
selector = SelectKBest(f_regression, k=k_features)
X_all = pd.DataFrame(
    selector.fit_transform(X_all, y_all),
    columns=X_all.columns[selector.get_support()],
    index=X_all.index
)

# 3. Remove highly correlated features (lower threshold)
corr_matrix = X_all.corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper_tri.columns if any(upper_tri[column] > 0.98)]  # Changed from 0.95 to 0.98
if to_drop:
    print(f"Removing {len(to_drop)} highly correlated features")
    X_all = X_all.drop(columns=to_drop)

print(f"✅ Final feature count: {X_all.shape[1]} (Original: {len(var_threshold.get_support())})")
print(f"Feature/Sample Ratio: {X_all.shape[1]}/{len(X_all)} = {X_all.shape[1]/len(X_all):.3f}")

# === New: Use Isolation Forest for Outlier Detection and Handling ===
print("\n" + "="*80)
print("🔍 Isolation Forest Outlier Detection and Handling")
print("="*80)

from sklearn.ensemble import IsolationForest

# Train Isolation Forest
iso_forest = IsolationForest(
    contamination=0.1,  # Assume 10% of data are outliers
    random_state=RANDOM_SEED,
    n_estimators=100,
    max_samples='auto',
    n_jobs=N_JOBS,
    verbose=0
)

# Detect outliers (-1 indicates outlier, 1 indicates normal)
anomaly_labels = iso_forest.fit_predict(X_all)
anomaly_scores = iso_forest.score_samples(X_all)  # Anomaly score (lower is more anomalous)

# Statistics
n_anomalies = (anomaly_labels == -1).sum()
anomaly_ratio = n_anomalies / len(X_all)
print(f"\nOutlier Statistics:")
print(f"  Detected outliers: {n_anomalies} / {len(X_all)} ({anomaly_ratio:.2%})")
print(f"  Anomaly score range: [{anomaly_scores.min():.4f}, {anomaly_scores.max():.4f}]")

# Strategy 1: Create outlier indicator feature
X_all['is_anomaly'] = (anomaly_labels == -1).astype(int)
X_all['anomaly_score'] = anomaly_scores

# Strategy 2: Winsorization on original features of outliers (scale to reasonable range)
print(f"\nOutlier Handling Strategy:")
print(f"  ✅ Added outlier indicator feature (is_anomaly)")
print(f"  ✅ Added anomaly score feature (anomaly_score)")

# Scale extreme feature values of outlier samples (retain anomaly info but reduce extreme impact)
anomaly_indices = np.where(anomaly_labels == -1)[0]
if len(anomaly_indices) > 0:
    print(f"  ✅ Winsorization on {len(anomaly_indices)} outlier samples")
    
    # For each feature, scale extreme values in outlier samples to 5% and 95% quantiles
    feature_cols = [col for col in X_all.columns if col not in ['is_anomaly', 'anomaly_score']]
    
    for col in feature_cols:
        # Calculate quantiles of normal samples
        normal_values = X_all.loc[anomaly_labels == 1, col]
        if len(normal_values) > 0:
            lower_bound = normal_values.quantile(0.05)
            upper_bound = normal_values.quantile(0.95)
            
            # Scale outlier samples (clip to reasonable range)
            X_all.loc[anomaly_indices, col] = X_all.loc[anomaly_indices, col].clip(
                lower=lower_bound,
                upper=upper_bound
            )

print(f"  ✅ Outlier handling completed (not deleted, all samples retained)")
print(f"\nFinal feature count: {X_all.shape[1]} (including outlier indicator features)")

# === Improvement 3: Time-based Split ===
print("\n" + "="*80)
print("📊 Data Split")
print("="*80)

split_idx = int(len(X_all) * 0.8)
X_train_df = X_all.iloc[:split_idx].reset_index(drop=True)
X_val_df = X_all.iloc[split_idx:].reset_index(drop=True)
y_train = y_all.iloc[:split_idx].reset_index(drop=True)
y_val = y_all.iloc[split_idx:].reset_index(drop=True)

print(f"\nTraining samples: {X_train_df.shape}, Validation samples: {X_val_df.shape}")

# Standardization
scaler_train = StandardScaler()
X_train_scaled = pd.DataFrame(scaler_train.fit_transform(X_train_df), columns=X_train_df.columns)
X_val_scaled = pd.DataFrame(scaler_train.transform(X_val_df), columns=X_val_df.columns)

# === Improvement 4: Hyperparameter Tuning ===
print("\n" + "="*80)
print("🔬 Hyperparameter Tuning")
print("="*80)

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error

# Ridge Hyperparameter Search
print("\nTuning Ridge...")
best_ridge_alpha = 1.0
best_ridge_score = float('inf')
for alpha in [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]:
    ridge_temp = Ridge(alpha=alpha, random_state=RANDOM_SEED)
    ridge_temp.fit(X_train_scaled, y_train)
    pred = ridge_temp.predict(X_val_scaled)
    mse = mean_squared_error(y_val, pred)
    if mse < best_ridge_score:
        best_ridge_score = mse
        best_ridge_alpha = alpha
print(f"✅ Ridge Best alpha: {best_ridge_alpha}, MSE: {best_ridge_score:.6f}")

ridge = Ridge(alpha=best_ridge_alpha, random_state=RANDOM_SEED)
ridge.fit(X_train_scaled, y_train)

# === Improvement 5: LightGBM Optimized Parameters ===
print("\nTuning LightGBM...")
lgbm = lgb.LGBMRegressor(
    n_estimators=1000,  # Increase max trees
    learning_rate=0.01,  # Decrease learning rate
    max_depth=6,  # Increase depth
    num_leaves=31,  # Increase leaves
    min_child_samples=20,  # Decrease min child samples
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,  # Decrease regularization
    reg_lambda=0.1,
    min_split_gain=0.001,  # Decrease split gain threshold
    random_state=RANDOM_SEED,
    verbose=-1,
    n_jobs=N_JOBS,
    force_row_wise=True  # Avoid warning
)
lgbm.fit(
    X_train_df, y_train, 
    eval_set=[(X_val_df, y_val)],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)
print(f"✅ LightGBM Completed (Using {lgbm.best_iteration_ if hasattr(lgbm, 'best_iteration_') else lgbm.n_estimators} trees)")

# === Improvement 6: XGBoost Optimized Parameters ===
print("\nTuning XGBoost...")
xgb_model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=6,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,  # Decrease regularization
    reg_lambda=0.1,
    gamma=0.01,
    random_state=RANDOM_SEED,
    verbosity=0,
    n_jobs=N_JOBS,
    early_stopping_rounds=50,
    tree_method='hist'  # Accelerate training
)
xgb_model.fit(
    X_train_df, y_train, 
    eval_set=[(X_val_df, y_val)],
    verbose=False
)
print(f"✅ XGBoost Completed (Using {xgb_model.best_iteration if hasattr(xgb_model, 'best_iteration') else xgb_model.n_estimators} trees)")

# === Improvement 7: Stacking Ensemble ===
print("\n🔄 Building Stacking Ensemble...")
train_stack = np.column_stack([
    ridge.predict(X_train_scaled),
    lgbm.predict(X_train_df),
    xgb_model.predict(X_train_df)
])
val_stack = np.column_stack([
    ridge.predict(X_val_scaled),
    lgbm.predict(X_val_df),
    xgb_model.predict(X_val_df)
])

# Meta-learner uses Ridge
meta = Ridge(alpha=0.5, random_state=RANDOM_SEED)
meta.fit(train_stack, y_train)
val_pred = meta.predict(val_stack)

print(f"Meta-learner weights: {meta.coef_}")
print("Meta-learner completed")

# === Evaluation ===
val_rmse = np.sqrt(np.mean((val_pred - y_val)**2))
val_mae = np.mean(np.abs(val_pred - y_val))
val_r2 = 1 - np.sum((val_pred - y_val)**2) / np.sum((y_val - y_val.mean())**2)
direction_acc = np.mean(np.sign(val_pred) == np.sign(y_val))

# Calculate performance of each base model
ridge_pred_val = ridge.predict(X_val_scaled)
lgbm_pred_val = lgbm.predict(X_val_df)
xgb_pred_val = xgb_model.predict(X_val_df)

print(f"\n" + "="*80)
print(f"📊 Model Evaluation")
print("="*80)
print(f"\nBase Model Performance:")
print(f"  Ridge  - RMSE: {np.sqrt(mean_squared_error(y_val, ridge_pred_val)):.6f}")
print(f"  LGBM   - RMSE: {np.sqrt(mean_squared_error(y_val, lgbm_pred_val)):.6f}")
print(f"  XGB    - RMSE: {np.sqrt(mean_squared_error(y_val, xgb_pred_val)):.6f}")

print(f"\nEnsemble Model Performance:")
print(f"  Validation RMSE: {val_rmse:.6f}")
print(f"  Validation MAE: {val_mae:.6f}")
print(f"  Validation R²: {val_r2:.4f}")
print(f"  Direction Accuracy: {direction_acc:.2%}")

print(f"\nPrediction Distribution:")
print(f"  Prediction Mean: {val_pred.mean():.6f}")
print(f"  Prediction Std Dev: {val_pred.std():.6f}")
print(f"  Prediction Range: [{val_pred.min():.6f}, {val_pred.max():.6f}]")
print(f"  Actual Range: [{y_val.min():.6f}, {y_val.max():.6f}]")

# === Save Winsorization Bounds for Test Set ===
print(f"\n💾 Saving outlier handling parameters...")
winsorize_bounds = {}
feature_cols = [col for col in X_all.columns if col not in ['is_anomaly', 'anomaly_score']]
normal_mask = (anomaly_labels == 1)

for col in feature_cols:
    normal_values = X_all.loc[normal_mask, col]
    if len(normal_values) > 0:
        winsorize_bounds[col] = (
            normal_values.quantile(0.05),
            normal_values.quantile(0.95)
        )

print(f"✅ Saved Winsorization bounds for {len(winsorize_bounds)} features")

# Save models (including Isolation Forest)
models = {
    'ridge': ridge,
    'lgbm': lgbm,
    'xgb': xgb_model,
    'meta_learner': meta,
    'scaler': scaler_train,
    'feature_names': X_train_df.columns.tolist(),  # Save feature names
    'iso_forest': iso_forest,  # Save Isolation Forest
    'winsorize_bounds': winsorize_bounds,
    'imputer': imputer,
    'raw_feature_names': raw_feature_names,
    'raw_feature_medians': raw_feature_medians
}

print('\n✅ Optimized training process completed')
print(f'Model dictionary keys: {list(models.keys())}')

## ✨ Isolation Forest Implementation Highlights

### Core Code Logic

```python
# 1. Train Isolation Forest
iso_forest = IsolationForest(contamination=0.1, n_estimators=100)
anomaly_labels = iso_forest.fit_predict(X_all)  # -1=Outlier, 1=Normal
anomaly_scores = iso_forest.score_samples(X_all)  # Anomaly Score

# 2. Add Anomaly Features
X_all['is_anomaly'] = (anomaly_labels == -1).astype(int)
X_all['anomaly_score'] = anomaly_scores

# 3. Winsorization (Do not delete, only scale)
for col in features:
    lower = normal_values.quantile(0.05)
    upper = normal_values.quantile(0.95)
    X_all.loc[anomaly_indices, col] = X_all.loc[anomaly_indices, col].clip(lower, upper)
```

### Why Not Delete Outliers?

1. **Financial Data Characteristics** - Extreme events (like market crashes) contain important information
2. **Avoid Data Leakage** - Test sets may contain outliers, models need to handle them
3. **Retain Sample Size** - Deleting data reduces training samples, affecting model performance
4. **Pattern Learning** - Allow the model to learn how to handle anomalous situations

### Test Set Processing

The test set will use the same `iso_forest` and `winsorize_bounds` for consistent processing, ensuring consistency in the training-prediction pipeline.

## 3. (Optional) Load an Explicit Trusted Model

Disabled by default. Pickle can execute code: use only a model you trust, with the working notebook model schema.


In [ ]:
MODEL_PATH = os.getenv("HULL_MODEL_PATH", "")
if MODEL_PATH:
    if os.getenv("HULL_ALLOW_PICKLE") != "1":
        raise ValueError("Set HULL_ALLOW_PICKLE=1 only if you trust HULL_MODEL_PATH.")
    with open(Path(MODEL_PATH).expanduser(), "rb") as f:
        loaded_models = pickle.load(f)
    required_keys = {
        "ridge", "lgbm", "xgb", "meta_learner", "scaler", "feature_names",
        "iso_forest", "winsorize_bounds", "imputer", "raw_feature_names", "raw_feature_medians"
    }
    if not isinstance(loaded_models, dict) or not required_keys.issubset(loaded_models):
        raise ValueError("Trusted pickle does not match the working notebook model schema.")
    models = loaded_models
    print("Loaded explicitly selected trusted model.")
else:
    print("Using the model trained above; no pickle file is loaded.")


## 4. Load Test Data

In [ ]:
# Load Test Data
print("=" * 80)
print("📊 Load Test Data")
print("=" * 80)

# TEST_PATH comes from the configuration cell.

print(f"Using test data path: {TEST_PATH}\n")

try:
    test = pd.read_csv(TEST_PATH)
    
    print(f"✅ Test set loaded successfully!")
    print(f"Test set shape: {test.shape}")
    print(f"Test set columns: {test.columns.tolist()}")
    print(f"\nFirst 5 rows of test set:")
    print(test.head())
    
    # Save row_id for final submission
    if 'row_id' in test.columns:
        row_ids = test['row_id']
        print(f"\n✓ Using 'row_id' column as Row ID")
    elif 'batch_id' in test.columns:
        row_ids = test['batch_id']
        print(f"\n✓ Using 'batch_id' column as Row ID")
    else:
        row_ids = pd.Series(range(len(test)), name='row_id')
        print(f"\n⚠️  row_id/batch_id column not found, using index as ID")
    
    print(f"Number of Row IDs: {len(row_ids)}")
    
except Exception as e:
    print(f"\n❌ Failed to load test data: {e}")
    import traceback
    traceback.print_exc()
    raise

## 5. Data Preprocessing Function

In [ ]:
def preprocess_test_data(test_df):
    """Reuse fitted training imputation, then select features and detect anomalies."""
    if not isinstance(test_df, pd.DataFrame):
        if hasattr(test_df, "to_dicts"):
            test_df = pd.DataFrame(test_df.to_dicts())
        else:
            test_df = test_df.to_pandas() if hasattr(test_df, "to_pandas") else pd.DataFrame(test_df)
    X_test = test_df.copy(deep=True).reset_index(drop=True)
    rename_map = {
        "lagged_market_forward_excess_returns": "market_forward_excess_returns",
        "lagged_risk_free_rate": "risk_free_rate"
    }
    # Preserve the historical mapping; its timing mismatch is documented.
    X_test = X_test.rename(columns={k: v for k, v in rename_map.items()
                                   if k in X_test.columns and v not in X_test.columns})
    raw_names = models["raw_feature_names"]
    raw = X_test.reindex(columns=raw_names).astype(float)
    raw = raw.replace([np.inf, -np.inf], np.nan)
    if models.get("imputer") is not None:
        raw = pd.DataFrame(models["imputer"].transform(raw), columns=raw_names)
    else:
        raw = raw.fillna(models["raw_feature_medians"])
    if not np.isfinite(raw.to_numpy()).all():
        raise ValueError("Test features remain non-finite after training-based imputation.")
    selected_names = [f for f in models["feature_names"] if f not in ["is_anomaly", "anomaly_score"]]
    aligned = raw[selected_names].copy()
    labels = models["iso_forest"].predict(aligned)
    scores = models["iso_forest"].score_samples(aligned)
    aligned["is_anomaly"] = (labels == -1).astype(int)
    aligned["anomaly_score"] = scores
    anomaly_mask = labels == -1
    for col, (lower, upper) in models["winsorize_bounds"].items():
        if col in aligned.columns:
            aligned.loc[anomaly_mask, col] = aligned.loc[anomaly_mask, col].clip(lower, upper)
    return aligned.reindex(columns=models["feature_names"])

print("Preprocessing function ready.")


## 6. Prediction Function

In [ ]:
def make_predictions(X_test, models_dict):
    """
    Make predictions using the ensemble model
    """
    # Ridge requires standardization
    X_test_scaled = models_dict['scaler'].transform(X_test)
    ridge_pred = models_dict['ridge'].predict(X_test_scaled)
    
    # LightGBM and XGBoost use original data (tree models are insensitive to scaling)
    lgbm_pred = models_dict['lgbm'].predict(X_test)
    xgb_pred = models_dict['xgb'].predict(X_test)
    
    # Stacking predictions
    stacked_features = np.column_stack([ridge_pred, lgbm_pred, xgb_pred])
    
    # Meta-learner prediction
    final_predictions = models_dict['meta_learner'].predict(stacked_features)
    
    return final_predictions

print("✅ Prediction function definition completed")

## 7. Generate Prediction Results

In [ ]:
print("=" * 80)
print("🚀 Start Prediction Process")
print("=" * 80)

try:
    # Preprocess test data
    print("\nStep 1/3: Preprocess test data...")
    X_test = preprocess_test_data(test)
    print(f"✅ Preprocessing completed, feature shape: {X_test.shape}")
    
    # Generate predictions
    print("\nStep 2/3: Generate predictions...")
    predictions = make_predictions(X_test, models)
    print(f"✅ Prediction completed!")
    
    # Output prediction statistics
    print("\nStep 3/3: Prediction Result Statistics")
    print(f"  Prediction Count: {len(predictions)}")
    print(f"  Mean: {predictions.mean():.6f}")
    print(f"  Std Dev: {predictions.std():.6f}")
    print(f"  Min: {predictions.min():.6f}")
    print(f"  Max: {predictions.max():.6f}")
    print(f"  Median: {np.median(predictions):.6f}")
    
    # Check outliers
    positive_pct = (predictions > 0).sum() / len(predictions) * 100
    print(f"\n  Positive Prediction Ratio: {positive_pct:.2f}%")
    print(f"  Negative Prediction Ratio: {100-positive_pct:.2f}%")
    
except Exception as e:
    print(f"\n❌ Prediction failed: {e}")
    import traceback
    traceback.print_exc()
    raise

## 8. Convert to Portfolio Weights

According to competition requirements, convert predicted returns to portfolio weights [0, 2]

In [ ]:
def convert_to_weights(predictions, method='enhanced_sigmoid'):
    """
    Convert predicted returns to portfolio weights [0, 2]
    
    Parameters:
        predictions: Predicted returns
        method: Conversion method
            - 'enhanced_sigmoid': Enhanced sigmoid, better utilization of prediction range
            - 'adaptive': Adaptive strategy
            - 'percentile': Percentile-based mapping
            - 'threshold': Threshold-based discrete weights
    """
    if method == 'enhanced_sigmoid':
        # Use standardized scores of predictions
        pred_mean = np.mean(predictions)
        pred_std = np.std(predictions)
        
        if pred_std > 0:
            # Standardization
            z_scores = (predictions - pred_mean) / pred_std
            # Use sigmoid to map to [0, 2], adjust steepness
            weights = 2 / (1 + np.exp(-2 * z_scores))
        else:
            # If std dev is 0, return neutral weights
            weights = np.ones_like(predictions)
        
    elif method == 'adaptive':
        # Adaptive strategy: Based on relative position of predictions
        pred_min = np.min(predictions)
        pred_max = np.max(predictions)
        
        if pred_max > pred_min:
            # Linear scaling to [0, 2]
            weights = 2 * (predictions - pred_min) / (pred_max - pred_min)
        else:
            weights = np.ones_like(predictions)
    
    elif method == 'percentile':
        # Percentile-based mapping
        percentiles = np.percentile(predictions, [20, 40, 60, 80])
        weights = np.zeros_like(predictions)
        
        weights[predictions <= percentiles[0]] = 0.0
        weights[(predictions > percentiles[0]) & (predictions <= percentiles[1])] = 0.5
        weights[(predictions > percentiles[1]) & (predictions <= percentiles[2])] = 1.0
        weights[(predictions > percentiles[2]) & (predictions <= percentiles[3])] = 1.5
        weights[predictions > percentiles[3]] = 2.0
        
    elif method == 'threshold':
        # Median-based binary strategy
        median = np.median(predictions)
        weights = np.where(predictions > median, 1.5, 0.5)
        
    else:
        raise ValueError(f"Unknown method: {method}")
    
    # Ensure within [0, 2] range
    weights = np.clip(weights, 0.0, 2.0)
    return weights

# Convert to weights - Use enhanced sigmoid strategy
weights = convert_to_weights(predictions, method='enhanced_sigmoid')

print(f"Weight Statistics:")
print(f"  Mean: {weights.mean():.4f}")
print(f"  Std Dev: {weights.std():.4f}")
print(f"  Min: {weights.min():.4f}")
print(f"  Max: {weights.max():.4f}")
print(f"  Median: {np.median(weights):.4f}")
print(f"  Range Check: [{weights.min():.2f}, {weights.max():.2f}] (Should be in [0, 2])")

# Weight Distribution
print(f"\nWeight Distribution:")
print(f"  [0.0-0.5): {np.sum((weights >= 0.0) & (weights < 0.5))}")
print(f"  [0.5-1.0): {np.sum((weights >= 0.5) & (weights < 1.0))}")
print(f"  [1.0-1.5): {np.sum((weights >= 1.0) & (weights < 1.5))}")
print(f"  [1.5-2.0]: {np.sum((weights >= 1.5) & (weights <= 2.0))}")

## 9. Generate Submission File

In [ ]:
print("=" * 80)
print("📝 Generate Submission File")
print("=" * 80)

try:
    # Create submission DataFrame
    # Note: Column names may need adjustment based on competition requirements
    submission = pd.DataFrame({
        'row_id': row_ids,  
        'prediction': weights
    })
    
    print("\nSubmission File Preview:")
    print(submission.head(10))
    print(f"\nSubmission File Shape: {submission.shape}")
    print(f"Submission File Columns: {submission.columns.tolist()}")
    
    # Check data quality
    print("\nData Quality Check:")
    print(f"  ✓ Missing Values: {submission.isnull().sum().sum()}")
    print(f"  ✓ Duplicate Rows: {submission.duplicated().sum()}")
    print(f"  ✓ Unique row_id: {submission['row_id'].nunique()}/{len(submission)}")
    
    # Save submission file
    if submission['row_id'].isna().any() or submission['row_id'].duplicated().any():
        raise ValueError('Submission row IDs must be present and unique.')
    if not np.isfinite(weights).all() or not ((weights >= 0) & (weights <= 2)).all():
        raise ValueError('Submission weights must be finite and within [0, 2].')
    submission_path = OUTPUT_DIR / 'submission.csv'
    submission.to_csv(submission_path, index=False)
    print(f"\n✅ Submission file saved: {submission_path}")
    
    # Verify file size
    import os
    file_size = submission_path.stat().st_size
    print(f"File Size: {file_size:,} bytes ({file_size/1024:.2f} KB)")
    
except Exception as e:
    print(f"\n❌ Failed to generate submission file: {e}")
    import traceback
    traceback.print_exc()
    raise

In [ ]:
# === Define predict function for Kaggle Inference Server and start server
print("="*80)
print("🚀 Preparing Inference Server")
print("="*80)

# Only import and run inference server in Kaggle environment
try:
    if not (os.getenv('HULL_RUN_KAGGLE_SERVER') == '1' or os.getenv('KAGGLE_IS_COMPETITION_RERUN')):
        raise ImportError('Kaggle server disabled; enable explicitly for a Kaggle gateway run.')
    from kaggle_evaluation.default_inference_server import DefaultInferenceServer
    
    print("✅ Kaggle environment detected, preparing inference server...")
    
    # Reuse previously defined preprocess_test_data, make_predictions, and convert_to_weights
    def predict(test_batch):
        """Function called by Kaggle Inference Server.
        Args: test_batch - pandas or Polars DataFrame (single batch)
        Returns: float (0-2)
        """
        try:
            # Preprocess (consistent with training)
            X = preprocess_test_data(test_batch)
            # Generate predictions using trained models
            preds = make_predictions(X, models)
            
            # Convert predicted returns to portfolio weights [0, 2]
            # Use more aggressive adaptive strategy
            if hasattr(preds, '__len__') and len(preds) > 0:
                # Convert each prediction to weight (steeper sigmoid)
                weights = 2 / (1 + np.exp(-np.clip(20 * preds, -700, 700)))  # Changed from 10 to 20
                signal = float(np.mean(weights))
            else:
                # Single prediction value
                signal = float(2 / (1 + np.exp(-np.clip(20 * preds, -700, 700))))
            
            # Ensure within [0, 2] range
            signal = np.clip(signal, 0.0, 2.0)
            return float(signal)
            
        except Exception as e:
            print(f"Prediction error: {e}")
            import traceback
            traceback.print_exc()
            return 1.0  # Return neutral position
    
    # Create and run inference server
    inference_server = DefaultInferenceServer(predict)
    
    if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
        print('\n🏁 Competition Rerun - Starting serve()')
        inference_server.serve()
    else:
        print('\n🧪 Kaggle Environment Local Test - Running local gateway')
        inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))
    
    print('\n✅ Inference Server Started')

except ImportError:
    print("Kaggle inference server skipped (disabled or evaluation package unavailable).")
    print("✅ submission.csv generation completed in local environment, available for local inspection; competition acceptance is not validated")